[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C16/E16-2_Laminoir.ipynb)

# Cage de laminoir : prédire le torseur des efforts de réaction

## Le contexte

Le même banc de laminage, vu de l'embase de la cage. Le bureau d'études dimensionne les fixations et le massif avec le torseur des efforts que la cage transmet à son embase : trois forces et trois moments. Ces efforts viennent du serrage, de l'écartement réel des cylindres, qui dépend du jeu de la vis et du cédage de la cage, et d'un frottement qui change de signe selon le sens du dernier serrage. Ils dépendent donc d'événements passés plus ou moins loin : le dernier changement de consigne, le dernier sens de serrage. Le banc a relevé les mêmes 120 séquences de laminage que dans la section précédente ; cette fois, la cible est le torseur.

Les grandeurs à prédire sont six nombres à chaque pas de temps : c'est un problème de régression à plusieurs sorties, sur des séries temporelles.

Défi : prédire les six composantes du torseur et obtenir un R² moyen supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est la régression linéaire sur décalages de la section précédente : pour chaque pas, elle lit les 64 pas précédents (128 secondes) des trois signaux ; il atteint un R² moyen de 0,88. À vous de faire mieux avec un réseau à attention, qui choisit lui-même, à chaque pas, quels instants du passé regarder. La règle de découpage est fixée plus bas, et c'est celle du défi précédent : la base de test est faite de séquences entières. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Laminoir.xlsx`, dans le dossier `donnees` du dépôt : 14 400 lignes, 24 colonnes. Une ligne est un pas de 2 secondes d'une séquence de laminage de 4 minutes. C'est le jeu de la section précédente, avec les autres sorties.

| Colonne | Signification | Unité |
|---|---|---|
| `sequence` | identifiant de la séquence, de `s01` à `s120`, sert à découper et à ranger les pas, n'est pas une entrée | catégorie |
| `temps_s` | temps depuis le début de la séquence, pas de 2 s, sert à ranger les pas | s |
| `nuance` | nuance de la bande : `doux`, `mi_dur` ou `dur` | catégorie |
| `pression_MPa` | pression mesurée dans le vérin de serrage | MPa |
| `temperature_huile_C` | température mesurée de l'huile du vérin | °C |
| `consigne_ecartement_mm` | consigne d'écartement des cylindres | mm |
| `sigma_xx_MPa` à `eps_yz_um_m` | contraintes et déformations à l'alésage du vérin, cibles de la section précédente, écartées ici | MPa et µm/m |
| `force_x_kN`, `force_y_kN`, `force_z_kN` | forces de réaction à l'embase | kN |
| `moment_x_kNm`, `moment_y_kNm`, `moment_z_kNm` | moments de réaction à l'embase | kN·m |

Origine : jeu simulé, cage de laminoir (cédage de la cage, jeu de la vis, frottement selon le sens du dernier serrage, plasticité du vérin). Quelques températures d'huile ne sont pas relevées. Il enseigne une méthode, et aucune conclusion de laminage n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la longueur des séquences avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Laminoir.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Laminoir.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cibles = [c for c in df.columns if c.startswith(("force_", "moment_"))]
print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides :", vides[vides > 0].to_dict())
longueurs = df.groupby("sequence").size()
print("séquences :", len(longueurs), "de", longueurs.min(), "à", longueurs.max(), "pas")
print("sorties à prédire :", cibles)
df.head(3)

Lecture du résultat : 14 400 lignes, 24 colonnes, 120 séquences de 120 pas. La température de l'huile compte 276 cases vides, qu'il faudra remplir. Les six sorties à prédire sont trois forces et trois moments.

### Reproductibilité

Un réseau de neurones part de poids tirés au hasard et lit ses exemples dans un ordre tiré au hasard. La cellule suivante fixe toutes les graines, demande des opérations de calcul déterministes, impose un seul fil de calcul et un jeu d'instructions donné. Avec elle, deux exécutions du notebook sur le même ordinateur donnent les mêmes poids et le même score. Chaque cellule qui construit un réseau rappelle la graine dans sa première ligne : on peut la rejouer seule sans changer le résultat. Sur un autre ordinateur, ou sur un processeur qui mêle deux types de cœurs, les arrondis peuvent différer et le score changer à la troisième décimale.

In [ ]:
# Réseaux de neurones : mêmes résultats à chaque exécution.
# À placer après la cellule de lecture (qui définit GRAINE),
# avant tout usage de TensorFlow.
import logging
import os

# Lues à l'import de TensorFlow : journal réduit, oneDNN limité
# au produit matriciel, jeu d'instructions fixé.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["ONEDNN_MAX_CPU_ISA"] = "AVX2"
logging.getLogger("tensorflow").setLevel(logging.ERROR)   # avertissements TensorFlow coupés avant l'import

import numpy as np
import tensorflow as tf
import keras

tf.get_logger().setLevel("ERROR")
keras.backend.clear_session()      # session propre avant toute construction

# Graines, opérations déterministes et un seul fil de calcul.
keras.utils.set_random_seed(GRAINE)
tf.config.experimental.enable_op_determinism()
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| NumPy", np.__version__)

### Étape 2 : mettre le test de côté

Comme dans la section précédente, on tire des séquences entières : un quart des séquences, avec leurs 120 pas, est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. Le groupe est l'identifiant de la séquence, la graine est 42 : les deux défis du jeu partagent la même base de test, sans fuite possible puisque chacun n'apprend que sur les séquences d'apprentissage. Dans l'apprentissage restant, un cinquième des séquences est réservé à la validation : il servira à comparer les modèles, jamais le test. Les contraintes et les déformations, cibles du défi précédent, ne servent pas d'entrées.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

entrees = ["sequence", "nuance", "pression_MPa", "temperature_huile_C", "consigne_ecartement_mm"]
X, y = df[entrees], df[cibles]
i_train, i_test = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=GRAINE).split(X, y, groups=df["sequence"]))
X_train, X_test, y_train, y_test = X.iloc[i_train], X.iloc[i_test], y.iloc[i_train], y.iloc[i_test]
i_fit, i_val = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=GRAINE).split(X_train, y_train, groups=X_train["sequence"]))
X_fit, X_val, y_fit, y_val = X_train.iloc[i_fit], X_train.iloc[i_val], y_train.iloc[i_fit], y_train.iloc[i_val]
for nom, partie in [("ajustement", X_fit), ("validation", X_val), ("test", X_test)]:
    print(f"{nom} : {partie['sequence'].nunique()} séquences, {len(partie)} pas")

### Étape 3 : regarder le torseur sur l'apprentissage

La figure place, pour trois séquences de l'apprentissage, une par nuance, la consigne d'écartement et deux composantes du torseur, `force_x_kN` et `moment_z_kNm`, en fonction du temps. Les paliers de consigne se succèdent.

In [ ]:
nuances = ["doux", "mi_dur", "dur"]
exemples = [X_train.loc[X_train["nuance"] == n, "sequence"].iloc[0] for n in nuances]
fig, axes = plt.subplots(3, 1, figsize=(6.4, 5.2), sharex=True)
for sequence, nuance, style, gris in zip(exemples, nuances, ["-", "--", ":"], ["0.0", "0.35", "0.6"]):
    pas = df[df["sequence"] == sequence]
    for ax, colonne in zip(axes, ["consigne_ecartement_mm", "force_x_kN", "moment_z_kNm"]):
        ax.plot(pas["temps_s"], pas[colonne], linestyle=style, color=gris, label=f"{sequence} ({nuance})")
for ax, nom in zip(axes, ["consigne (mm)", "force x (kN)", "moment z (kN·m)"]):
    ax.set_ylabel(nom)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("temps (s)")
axes[0].legend(frameon=False, ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

Lecture du résultat : la force x et le moment z ne suivent pas la consigne marche pour marche. La force x porte la traction de la bande, qui dépend de la nuance, et des à-coups à chaque changement de consigne ; le moment z change de niveau selon le sens du dernier serrage, c'est-à-dire d'un événement qui peut remonter à plusieurs paliers. Pour prédire ce moment à un instant, le modèle doit retrouver le dernier changement de consigne utile, quelle que soit sa distance : c'est ce que sait faire l'attention.

### Étape 4 : préparer les séquences

Chaque séquence devient un tableau de 120 pas et six variables : les trois signaux et la nuance en indicatrices. Une température non relevée reprend la dernière valeur lue dans la même séquence ; en tête, elle reçoit la médiane du signal sur l’ajustement seul. Les sorties sont rangées de la même façon, en six colonnes par pas. Les entrées et les sorties du réseau seront centrées et réduites avec la moyenne et l'écart type des séquences d'apprentissage seules.

In [ ]:
# Votre code :

### Étape 5 : construire le modèle de départ

Le modèle de départ est la régression linéaire sur décalages de la section précédente. Pour chaque pas, on lui donne les valeurs des trois signaux aux 64 pas précédents (128 secondes de passé, le premier pas répété avant le début) et la nuance : 195 variables. Un pipeline les met à l'échelle puis ajuste une régression ridge, sur les séquences d'apprentissage seules.

In [ ]:
# Votre code :

### Étape 6 : mesurer sur la base de test

Le modèle prédit des séquences qu'il n'a jamais vues. Le R² est calculé sur les six sorties à la fois : c'est la moyenne des six R², chaque sortie pesant autant. On lit aussi le R² de chaque composante. Un témoin sans apprentissage, qui répond toujours la moyenne de chaque sortie sur l'apprentissage, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.metrics import r2_score

score_depart = r2_score(y_test, y_pred)
score_temoin = r2_score(y_test, np.tile(y_train.mean().to_numpy(), (len(y_test), 1)))
print(f"R² moyen des six sorties : {score_depart:.3f}")
for nom, r in zip(cibles, r2_score(y_test, y_pred, multioutput="raw_values")):
    print(f"  {nom} : {r:.3f}")
print(f"R² du témoin (toujours la moyenne de l'apprentissage) : {score_temoin:.3f}")

Lecture du résultat : le R² moyen est de 0,88. La composante la moins bien prédite est `moment_x_kNm` (0,76). Le témoin, qui répond toujours la moyenne de l'apprentissage, n'explique rien : son R² est voisin de zéro, et négatif parce que cette moyenne n'est pas celle du test. C'est ce chiffre, mesuré sur des séquences que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : un réseau à attention

Le modèle de départ de l'étape 5 reste le score de référence du défi. Les cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première prépare les séquences pour Keras, avec l'échelle apprise sur les séquences d'apprentissage seules. La deuxième construit un réseau LSTM, dont la norme du gradient est écrêtée à 1, et un réseau à attention, et les compare sur la validation. Le réseau à attention donne à chaque pas un poids à chaque pas du passé, par un produit entre ce que le pas cherche et ce que les pas précédents offrent ; un masque interdit de regarder le futur. La troisième lit ces poids pour une séquence de validation. La quatrième compare les composantes du torseur sur la validation. La dernière mesure le réseau retenu une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

fig, ax = plt.subplots(figsize=(6.4, 3.4))
for t_i, style, gris in [(40, "--", "0.55"), (100, "-", "0.0")]:
    ax.plot(2 * np.arange(t_i + 1), poids[:, t_i, :t_i + 1].mean(axis=0), linestyle=style, color=gris, label=f"pas interrogé : {t_i} ({2 * t_i} s)")
ax.set(xlabel="instant regardé (s)", ylabel="poids d'attention moyen")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : sur la validation, la régression linéaire sur décalages atteint 0,83. Le LSTM à 64 unités atteint 0,93, le réseau à attention 0,93 : sur ces 18 séquences, les deux réseaux font à peu près jeu égal, bien au-dessus du modèle linéaire, et la validation retient « attention à 32 unités, 4 têtes, 2 blocs » à une faible différence. Le gain de l'attention n'est donc pas dans le score de cette validation ; il est dans ce que l'on peut lire du réseau. Les poids du premier bloc d'attention montrent où le réseau regarde : pour le pas 100, les 20 pas les plus récents portent 27 % du poids, et le reste se répartit sur tout le passé, y compris les 60 premiers pas, qui en portent 50 %. La figure montre que le poids est le plus élevé sur l'instant initial de la séquence (0,021) et sur les derniers instants (0,013 en moyenne), avec un creux entre les deux (0,008 en moyenne entre 20 et 120 s) : le réseau relit surtout l'état de départ et le passé récent, et moins ce qui s'est passé entre les deux. Mesuré une seule fois sur le test, le réseau retenu atteint un R² moyen de 0,95, soit 0,07 de plus que le modèle de départ.

## Le score de référence

Le score publié est le R² moyen du modèle de départ de l'étape 5 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, y_pred)
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.2f" % score)

## Synthèse

1. Le torseur dépend d'événements passés plus ou moins loin, comme le dernier changement de consigne ou le dernier sens de serrage : une fenêtre fixe de décalages les retrouve imparfaitement (0,88 sur le test ; sur la validation, la composante `force_y_kN` n'atteint que 0,63, contre 0,88 pour le réseau à attention).
2. L'attention donne à chaque pas un poids à chaque pas du passé, calculé à partir des signaux eux-mêmes ; un masque interdit de regarder le futur, et un numéro de position, appris, dit au réseau où il se trouve dans la séquence.
3. Le LSTM et le réseau à attention se comparent sur une validation prise dans l'apprentissage, en séquences entières (0,93 et 0,93 ici) ; le test ne sert qu'une fois, au réseau retenu (0,95 ici).
4. Les poids d'attention se lisent : ils montrent la part du passé récent et celle du passé lointain dans chaque prédiction.
5. La graine est rappelée dans la cellule qui construit chaque réseau : on la rejoue seule sans changer le résultat.